In [ ]:
from PIL import Image
import numpy as np
import os

In [ ]:
def generateMrea(metallic, roughness, emissive, ao, noiseScale = 0.1):
    if ao is None or ao == 1.0:
        img = Image.new('RGB', (16, 16), color = (int(metallic*255), int(roughness*255), int(emissive*255)))
    else:
        img = Image.new('RGBA', (16, 16), color = (int(metallic*255), int(roughness*255), int(emissive*255), int(ao*255)))
    pixels = img.load()
    noise = np.random.rand(16, 16)
    for x in range(img.size[0]):
        for y in range(img.size[1]):
            px = pixels[x,y]
            variedRoughness = max(0, min(1, noiseScale * noise[x][y] + roughness ))
            pxRoughness = int(variedRoughness*255)
            if len(px) == 4:
                pixels[x,y] = (px[0], pxRoughness, px[2], px[3])
            elif len(px) == 3:
                pixels[x,y] = (px[0], pxRoughness, px[2])
    return img

In [ ]:
metals = [
    "diamond_white_v0", "diamond_black_v0",
    "gold_white_v0", "gold_black_v0",
    "gold_chest_white_v0", "gold_chest_black_v0",
    "silver_white_v0", "silver_black_v0",
    "silver_chest_white_v0", "silver_chest_black_v0",
    "neptunium_white_v0", "neptunium_black_v0",
    "neptunium_chest_white_v0", "neptunium_chest_black_v0",
    ]
stones = [
    "basalt_white_v0", "basalt_black_v0",
    "basalt_brick_white_v0", "basalt_brick_black_v0",
    "bedrock_white_v0", "bedrock_black_v0",
    "clay_black_v0", "clay_white_v0",
    "clay_brick_black_v0", "clay_brick_white_v0",
    "clay_chiselled_black_v0", "clay_chiselled_white_v0",
    "cobblestone_white_v0", "cobblestone_black_v0",
    "cobblestone_brick_white_v0", "cobblestone_brick_black_v0",
    "cobblestone_chiselled_white_v0", "cobblestone_chiselled_black_v0",
    "limestone_white_v0", "limestone_black_v0",
    "limestone_brick_white_v0", "limestone_brick_black_v0",
    "limestone_chiselled_white_v0", "limestone_chiselled_black_v0",
    "stone_white_v0", "stone_black_v0",
    "stone_white_v1", "stone_black_v1",
    "stone_white_v2", "stone_black_v2",
    "stone_brick_white_v0", "stone_brick_black_v0",
    "stone_brick2_white_v0", "stone_brick2_black_v0",
    "stone_brick2_mossy_white_v0", "stone_brick2_mossy_black_v0",
    "stone_chiselled_white_v0", "stone_chiselled_black_v0",
    "stone_chiselled_mossy_white_v0", "stone_chiselled_mossy_black_v0",
    "terrazzo_white_v0", "terrazzo_black_v0",
    "terrazzo_brick_white_v0", "terrazzo_brick_black_v0",
    ]
smoothStones = [
    "basalt_polished_white_v0", "basalt_polished_black_v0",
    "clay_polished_white_v0", "clay_polished_black_v0",
    "granite_polished_white_v0", "granite_polished_black_v0",
    "stone_polished_white_v0", "stone_polished_black_v0"
    ]
default = ["shared/default"]

metalMrea = (1.0, 0.3, 0.0, 1.0)
stoneMrea = (0.0, 0.72, 0.0, 1.0)
smoothMrea = (0.0, 0.6, 0.0, 1.0)
defaultMrea = (0.0, 0.85, 0.0, 1.0)

def quant(*args):
    return [int(f*255) for f in args]

fullValues = [
    (metalMrea, 0.01, metals),
    (stoneMrea, 0.05, stones),
    (smoothMrea, 0.01, smoothStones),
    (defaultMrea, 0.025, default),
]

for (mrea, roughnessNoiseScale, blocks) in fullValues:
    for block in blocks:
        filename = "../../data/blocks/textures/{}_mrea.png".format(block)
        topFilename = "../../data/blocks/textures/{}_top.png".format(block)
        if not os.path.exists(topFilename):
            continue
        generateMrea(*mrea, roughnessNoiseScale).save(filename)

In [ ]:
ores = [
    "coal_ore_white_v0", "coal_ore_black_v0",
    "diamond_ore_black_v0", "diamond_ore_white_v0",
    "gold_ore_black_v0", "gold_ore_white_v0",
    "gold_ore_black_v1", "gold_ore_white_v1",
    "silver_ore_black_v0", "silver_ore_white_v0",
    "neptunium_ore_black_v0", "neptunium_ore_white_v0",
]

stoneColor = (105, 108, 116)
threshold = 50

for ore in ores:
    for side in ["top", "bottom", "side"]:
        sourceFilename = "../../data/blocks/textures/{}_{}.png".format(ore, side)
        if not os.path.exists(sourceFilename):
            continue
        source = Image.open("../../data/blocks/textures/{}_{}.png".format(ore, side))
        mreaTex = Image.new('RGB', (16, 16))
        noise = np.random.rand(16, 16)
        for x in range(16):
            for y in range(16):
                px = source.getpixel((x, y))
                colorDiff = abs(px[0] - stoneColor[0]) + abs(px[1] - stoneColor[1]) + abs(px[2] - stoneColor[2])
                if colorDiff > threshold:
                    mrea = list(metalMrea)
                    mrea[1] += noise[x][y] * 0.01
                    mreaTex.putpixel((x, y), tuple(quant(*mrea)[:3]))
                else:
                    mrea = list(stoneMrea[:])
                    mrea[1] += noise[x][y] * 0.05
                    mreaTex.putpixel((x, y), tuple(quant(*mrea)[:3]))
        mreaTex.save("../../data/blocks/textures/{}_mrea{}.png".format(ore, side))
